In [8]:
from pathlib import Path
import shutil
import sqlite3

In [9]:
master_db_path = Path("../databases/Master_Database.db").resolve()
#mosaic_folder = Path("../output mosaics/non-flats")
helper_path = Path("../py_files/store_zeropoint.py")
#export_folder = Path("../lab batches/zeropoints_nonflat_01")


mosaic_folder = Path("../output mosaics/flats")
export_folder = Path("../lab batches/zeropoints_flat_01")

master_conn = sqlite3.connect(master_db_path)

In [10]:
def get_zeropoint_work(master_conn):
    sql = """
        SELECT group_id, mosaic_name_f
        FROM observations
        WHERE mosaic_name_f IS NOT NULL
            AND zpt_mean_f IS NULL
            AND error_f IS NULL
        ORDER BY group_id
    """
    
    
#    sql = """
#       SELECT group_id, mosaic_name
#       FROM observations
#       WHERE mosaic_name IS NOT NULL
#         AND zpt_mean IS NULL
#         AND error IS NULL
#       ORDER BY group_id
#   """

    work_list = master_conn.execute(sql).fetchall()

    return work_list

In [11]:
def make_zeropoint_commands(group_id, filename):
       return f'''
   echo "Processing group {group_id}"

   if ( -d "$batch_dir/working/g{group_id}" ) then
       echo "Work folder already exists; skipping group {group_id}"
   else
       if ( ! -s "$batch_dir/input_mosaics/{filename}" ) then
           echo "Missing input: {filename}"
           exit 1
       endif

       mkdir "$batch_dir/working/g{group_id}"
       if ( $status != 0 ) exit 1

       cd "$batch_dir/working/g{group_id}"
       if ( $status != 0 ) exit 1

       python /mnt/64bin/find_zeropoint.py3 "../../input_mosaics/{filename}" >&! "$batch_dir/logs/g{group_id}.log"
       set calibration_status = $status

       if ( $calibration_status != 0 ) then
           rm -f zpts.log
           if ( $status != 0 ) exit 1
       endif

       python "$batch_dir/store_zeropoint.py" "{filename}" "$batch_dir/results.csv"
       if ( $status != 0 ) exit 1

       cd "$batch_dir"
       if ( $status != 0 ) exit 1
   endif
   '''

In [12]:
def write_zeropoint_batch(work_list, mosaic_folder, helper_path, export_folder):
       if not work_list:
           raise ValueError("No mosaics need zeropoints")

       if not helper_path.is_file():
           raise FileNotFoundError(helper_path)

       for group_id, filename in work_list:
           source_path = mosaic_folder / filename

           if not source_path.is_file():
               raise FileNotFoundError(source_path)

       export_folder.mkdir(parents=True)

       input_folder = export_folder / "input_mosaics"
       input_folder.mkdir()

       shutil.copy2(helper_path, export_folder / "store_zeropoint.py")

       output = [
           "#!/bin/csh -f",
           'set batch_dir = "$cwd"',
           "mkdir -p working logs",
           "if ( $status != 0 ) exit 1",
       ]

       for group_id, filename in work_list:
           source_path = mosaic_folder / filename
           destination_path = input_folder / filename

           shutil.copy2(source_path, destination_path)

           commands = make_zeropoint_commands(group_id, filename)
           output.append(commands)

       script_path = export_folder / "run_zeropoints.csh"

       with script_path.open("w", encoding="utf-8") as file:
           file.write("\n".join(output) + "\n")

       return script_path

In [13]:
work_list = get_zeropoint_work(master_conn)

print("Mosaics needing zeropoints:", len(work_list))

for row in work_list[:5]:
   print(row)

Mosaics needing zeropoints: 151
(1, 'at2026aeiz-26_10_13-east-16-V-flat-g1-mosaic.fits')
(2, 'at2026aesj-26_10_13-pt5m-B-flat-g2-mosaic.fits')
(3, 'at2026aesj-26_10_13-pt5m-V-flat-g3-mosaic.fits')
(4, 'at2026aesj-26_10_13-pt5m-R-flat-g4-mosaic.fits')
(5, 'at2026aesj-26_10_13-pt5m-I-flat-g5-mosaic.fits')


In [14]:
script_path = write_zeropoint_batch(
   work_list,
   mosaic_folder,
   helper_path,
   export_folder,
)

print("Script saved to:", script_path.resolve())
print("Number of mosaics:", len(work_list))

Script saved to: /Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/lab batches/zeropoints_flat_01/run_zeropoints.csh
Number of mosaics: 151
